# 청년 복지정보 크롤러 (지자체 카테고리)

## 개요
Selenium과 BeautifulSoup을 사용해 정부 복지정보 포털의 **지자체** 카테고리에서 청년 대상 복지 서비스 정보를 자동으로 수집하는 크롤러입니다. 1~134페이지 전체 항목을 순회하며 각 항목의 상세 페이지에 진입해 지자체명, 표 정보(지원주기 / 신청방법 / 제공유형), 지원대상, 선정기준, 서비스 내용, 전화문의, 근거법령, 서식자료, 최종수정일 등 세부 필드를 추출한 뒤 CSV / TSV / JSON 세 가지 형식으로 저장합니다.

> **저작권 안내**: 이 코드가 접근하는 실제 사이트 도메인은 저작권 문제로 `target_site`라는 이름으로 치환하였습니다.

## 주요 기능
1. **`extract_jijache_info`** — 상세 페이지에서 모든 필드를 한 번에 추출하는 핵심 함수입니다. 제목 / 서비스세부내용 / 지자체명 / 표 정보 / 섹션별 내용 / 전화문의 / 근거법령 / 서식자료 / 최종수정일까지 총 9단계로 순차 파싱합니다.
2. **표 정보 추출** — 지원주기·신청방법·제공유형처럼 헤더와 값이 같은 `cl-container` 안에 있는 표 형태 데이터를, 헤더 텍스트를 먼저 찾은 뒤 같은 컨테이너 내 다음 값을 매칭하는 방식으로 추출합니다.
3. **전화문의 추출** — 기관명(`blt-tit-m`)과 전화번호(`em-txt`)가 번갈아 나오는 구조를 순회하면서, 전화번호 패턴이 확인되면 직전 기관명과 묶어 저장합니다.
4. **서식/자료 추출** — 파일 확장자 정규식(`hwp`, `pdf`, `xlsx` 등)으로 실제 첨부파일만 골라내고, 전화번호 패턴과 겹치는 오탐은 제외합니다.
5. **`save_data`** — 수집된 데이터를 pandas `DataFrame`으로 변환한 뒤 CSV / TSV / JSON 세 형식으로 저장합니다.

## 코드 구조
- **유틸리티 함수** — 텍스트 정제(`clean_text`)
- **정보 추출 함수** — `extract_jijache_info`와 그 내부의 9단계 필드별 파싱 로직
- **메인 실행** — Selenium 드라이버 설정, 페이지(1~134)/항목 반복, 데이터 저장

## 설계 포인트
- **`line-tit` 기반 범용 섹션 탐색**: 지원대상 / 선정기준 / 서비스 내용 / 전화문의 / 근거법령 / 서식자료 등 대부분의 섹션이 "섹션 제목(`line-tit`) → 다음 형제 `cl-layout-wrap`" 패턴을 공유한다는 점을 이용해, 각 섹션마다 같은 탐색 로직을 재사용합니다.
- **중앙/민간 크롤러와의 차이**: 지자체 상세페이지는 탭 클릭 없이 한 페이지에 모든 정보가 있고, 표 형태 필드(지원주기 / 신청방법 / 제공유형)와 지자체명이 별도로 존재해 파싱 로직을 새로 작성했습니다.
- **대용량 수집 대비**: 134페이지 전체를 순회하는 만큼 페이지/항목 단위 예외 처리를 넣어 특정 항목에서 오류가 나도 크롤링 전체가 중단되지 않도록 했습니다.


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# ========================================
# 유틸리티 함수
# ========================================

def clean_text(text):
    """텍스트 정리"""
    if text is None:
        return ""
    text = text.replace("이 누리집은 대한민국 공식 전자정부 누리집입니다.", "")
    text = text.replace("[새창열림]링크 이동", "")
    text = text.replace("[새창열림]파일 미리보기", "")
    text = text.replace("미리보기", "")
    text = text.replace("다운로드", "")
    return ' '.join(text.split()).strip()

# ========================================
# 정보 추출 함수
# ========================================

def extract_jijache_info(driver, debug=False):
    """지자체 복지 정보 추출 - 완전 재작성"""

    data = {
        '상세URL': driver.current_url,
        '제목': '',
        '구분': '지자체'
    }

    try:
        # 페이지 로딩 대기
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.wlfare-info-nm"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # ===== 1. 제목 =====
        title_elem = soup.find('div', class_='wlfare-info-nm')
        if title_elem:
            title_text = title_elem.find('div', class_='cl-text')
            if title_text:
                data['제목'] = clean_text(title_text.get_text())
                if debug:
                    print(f"      [제목] {data['제목']}")

        # ===== 2. 서비스세부내용 (제목 바로 다음) =====
        all_htmlsnippets = soup.find_all('div', class_='cl-htmlsnippet')
        if all_htmlsnippets and len(all_htmlsnippets) > 0:
            intro_text = clean_text(all_htmlsnippets[0].get_text())
            if intro_text and len(intro_text) > 5:
                data['서비스세부내용'] = intro_text
                if debug:
                    print(f"      [서비스세부내용] {intro_text[:50]}...")

        # ===== 3. 지자체명 (diamond-blt) =====
        diamond_elems = soup.find_all('div', class_='diamond-blt')
        for diamond in diamond_elems:
            diamond_text_div = diamond.find('div', class_='cl-text')
            if diamond_text_div and '지자체' in clean_text(diamond_text_div.get_text()):
                # 같은 행의 다음 칸 찾기
                parent_row = diamond.find_parent('div', class_='cl-layout-content')
                if parent_row:
                    # 모든 output 찾기
                    outputs = parent_row.find_all('div', class_='cl-output')
                    for output in outputs:
                        if output != diamond:
                            value_div = output.find('div', class_='cl-text')
                            if value_div:
                                value = clean_text(value_div.get_text())
                                if value and value != '지자체' and len(value) > 2:
                                    data['지자체명'] = value
                                    if debug:
                                        print(f"      [지자체명] {value}")
                                    break
                if data.get('지자체명'):
                    break

        # ===== 4. 표 정보 (지원주기, 신청방법, 제공유형) =====
        # 헤더와 값이 같은 container 안에 있음
        table_fields = ['지원주기', '신청방법', '제공유형']

        for field in table_fields:
            # 헤더 찾기
            all_outputs = soup.find_all('div', class_='cl-output')
            for i, output in enumerate(all_outputs):
                text_div = output.find('div', class_='cl-text')
                if text_div and clean_text(text_div.get_text()) == field:
                    # 같은 container의 다음 output들 중에서 값 찾기
                    container = output.find_parent('div', class_='cl-container')
                    if container:
                        container_outputs = container.find_all('div', class_='cl-output')
                        for co in container_outputs:
                            if co != output:
                                co_text_div = co.find('div', class_='cl-text')
                                if co_text_div:
                                    value = clean_text(co_text_div.get_text())
                                    # 헤더가 아닌 실제 값인지 확인
                                    if value and value not in table_fields and len(value) < 50:
                                        data[f'표_{field}'] = value
                                        if debug:
                                            print(f"      [표_{field}] {value}")
                                        break
                    break

        # ===== 5. 섹션별 내용 (지원대상, 선정기준, 서비스 내용, 신청방법) =====
        sections = ['지원대상', '선정기준', '서비스 내용', '신청방법']

        for section_name in sections:
            if debug:
                print(f"      [{section_name}] 처리 중...")

            # line-tit 찾기
            line_tits = soup.find_all('div', class_='line-tit')
            for line_tit in line_tits:
                line_tit_text_div = line_tit.find('div', class_='cl-text')
                if not line_tit_text_div:
                    continue

                if clean_text(line_tit_text_div.get_text()) == section_name:
                    # line-tit의 부모 wrap
                    current_wrap = line_tit.find_parent('div', class_='cl-layout-wrap')
                    if current_wrap:
                        # 다음 wrap (형제)
                        next_wrap = current_wrap.find_next_sibling('div', class_='cl-layout-wrap')
                        if next_wrap:
                            # htmlsnippet 찾기
                            htmlsnippet = next_wrap.find('div', class_='cl-htmlsnippet')
                            if htmlsnippet:
                                content = clean_text(htmlsnippet.get_text())
                                if content and len(content) > 5:
                                    data[section_name] = content
                                    if debug:
                                        print(f"         ✓ {len(content)}자")
                                break

        # ===== 6. 전화문의 =====
        if debug:
            print(f"      [전화문의] 처리 중...")

        contacts = []
        line_tits = soup.find_all('div', class_='line-tit')
        for line_tit in line_tits:
            line_tit_text_div = line_tit.find('div', class_='cl-text')
            if line_tit_text_div and '전화문의' in clean_text(line_tit_text_div.get_text()):
                # line-tit을 포함한 wrap의 다음 형제 wrap 찾기
                current_wrap = line_tit.find_parent('div', class_='cl-layout-wrap')
                if current_wrap:
                    # 다음 wrap (margin-top: 16px)
                    next_wrap = current_wrap.find_next_sibling('div', class_='cl-layout-wrap')
                    if next_wrap:
                        # 그 안의 모든 cl-layout-wrap 찾기 (inline-block)
                        inner_wraps = next_wrap.find_all('div', class_='cl-layout-wrap', recursive=True)

                        current_org = None
                        for inner_wrap in inner_wraps:
                            # blt-tit-m 찾기 (기관명)
                            blt_elem = inner_wrap.find('div', class_='blt-tit-m')
                            if blt_elem:
                                blt_text_div = blt_elem.find('div', class_='cl-text')
                                if blt_text_div:
                                    current_org = clean_text(blt_text_div.get_text())

                            # em-txt 찾기 (전화번호)
                            em_elem = inner_wrap.find('div', class_='em-txt')
                            if em_elem:
                                em_text_div = em_elem.find('div', class_='cl-text')
                                if em_text_div:
                                    phone = clean_text(em_text_div.get_text())
                                    if re.search(r'\d{2,3}-\d{3,4}-\d{4}', phone) and current_org:
                                        combined = f"{current_org} {phone}"
                                        if combined not in contacts:
                                            contacts.append(combined)
                                        current_org = None  # 사용했으므로 초기화
                break

        if contacts:
            data['전화문의'] = '\n'.join(contacts)
            if debug:
                print(f"         ✓ {len(contacts)}개 연락처")

        # ===== 7. 근거법령 =====
        if debug:
            print(f"      [근거법령] 처리 중...")

        line_tits = soup.find_all('div', class_='line-tit')
        for line_tit in line_tits:
            line_tit_text_div = line_tit.find('div', class_='cl-text')
            if line_tit_text_div and '근거법령' in clean_text(line_tit_text_div.get_text()):
                # line-tit을 포함한 wrap의 다음 형제 wrap 찾기
                current_wrap = line_tit.find_parent('div', class_='cl-layout-wrap')
                if current_wrap:
                    # 다음 wrap (margin-top: 16px)
                    next_wrap = current_wrap.find_next_sibling('div', class_='cl-layout-wrap')
                    if next_wrap:
                        # 그 안의 blt-tit-m 찾기
                        blt_elem = next_wrap.find('div', class_='blt-tit-m')
                        if blt_elem:
                            blt_text_div = blt_elem.find('div', class_='cl-text')
                            if blt_text_div:
                                law = clean_text(blt_text_div.get_text())
                                if law:
                                    data['근거법령'] = law
                                    if debug:
                                        print(f"         ✓ {law[:50]}...")
                break

        # ===== 8. 서식/자료 =====
        if debug:
            print(f"      [서식/자료] 처리 중...")

        attachments = []
        line_tits = soup.find_all('div', class_='line-tit')
        for line_tit in line_tits:
            line_tit_text_div = line_tit.find('div', class_='cl-text')
            if line_tit_text_div and '서식' in clean_text(line_tit_text_div.get_text()):
                # line-tit을 포함한 wrap의 다음 형제 wrap 찾기
                current_wrap = line_tit.find_parent('div', class_='cl-layout-wrap')
                if current_wrap:
                    # 다음 wrap (margin-top: 16px)
                    next_wrap = current_wrap.find_next_sibling('div', class_='cl-layout-wrap')
                    if next_wrap:
                        # 그 안의 모든 blt-tit-m 찾기
                        blt_elements = next_wrap.find_all('div', class_='blt-tit-m')
                        for blt_elem in blt_elements:
                            blt_text_div = blt_elem.find('div', class_='cl-text')
                            if blt_text_div:
                                filename = clean_text(blt_text_div.get_text())
                                # 파일 확장자가 있는지 확인
                                if re.search(r'\.(hwp|hwpx|pdf|xlsx?|docx?|jpg|png|zip)$', filename, re.IGNORECASE):
                                    # 전화번호가 아닌지 확인
                                    if not re.search(r'\d{2,3}-\d{3,4}-\d{4}', filename):
                                        if filename not in attachments:
                                            attachments.append(filename)
                break

        if attachments:
            data['서식자료'] = '\n'.join(attachments)
            if debug:
                print(f"         ✓ {len(attachments)}개 파일")

        # ===== 9. 최종 수정일 =====
        if debug:
            print(f"      [최종수정일] 처리 중...")

        line_tits = soup.find_all('div', class_='line-tit')
        for line_tit in line_tits:
            line_tit_text_div = line_tit.find('div', class_='cl-text')
            if line_tit_text_div and '최종 수정일' in clean_text(line_tit_text_div.get_text()):
                # line-tit을 포함한 wrap의 다음 형제 wrap 찾기
                current_wrap = line_tit.find_parent('div', class_='cl-layout-wrap')
                if current_wrap:
                    # 다음 wrap (margin-top: 0px)
                    next_wrap = current_wrap.find_next_sibling('div', class_='cl-layout-wrap')
                    if next_wrap:
                        # 그 안의 blt-tit-m 찾기
                        blt_elem = next_wrap.find('div', class_='blt-tit-m')
                        if blt_elem:
                            blt_text_div = blt_elem.find('div', class_='cl-text')
                            if blt_text_div:
                                date = clean_text(blt_text_div.get_text())
                                if date:
                                    data['최종수정일'] = date
                                    if debug:
                                        print(f"         ✓ {date}")
                break

    except Exception as e:
        if debug:
            print(f"      ✗ 정보 추출 오류: {e}")

    return data

# ========================================
# 메인 실행
# ========================================

def save_data(data_list, filename_prefix, columns_order):
    """데이터 저장"""
    if not data_list:
        print(f"⚠️ 저장할 데이터가 없습니다")
        return []

    df = pd.DataFrame(data_list)
    existing_columns = [col for col in columns_order if col in df.columns]
    df = df.reindex(columns=existing_columns, fill_value='')

    saved_files = []

    # TSV
    tsv_file = f'{filename_prefix}.tsv'
    df.to_csv(tsv_file, index=False, sep='\t', encoding='utf-8-sig')
    print(f"✓ TSV 저장: {tsv_file}")
    saved_files.append(tsv_file)

    # CSV
    csv_file = f'{filename_prefix}.csv'
    df.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"✓ CSV 저장: {csv_file}")
    saved_files.append(csv_file)

    # JSON
    json_file = f'{filename_prefix}.json'
    df.to_json(json_file, orient='records', force_ascii=False, indent=2)
    print(f"✓ JSON 저장: {json_file}")
    saved_files.append(json_file)

    return saved_files

# ========================================
# 설정
# ========================================

print("="*70)
print("타겟사이트 지자체 복지정보 크롤러 시작")
print("="*70)

DEBUG_MODE = True
TEST_MODE = False

base_url = "https://www.target_site.go.kr/ssis-tbu/twataa/wlfareInfo/moveTWAT52005M.do"
TAB_ID = 2  # 지자체
START_PAGE = 1
END_PAGE = 134

COLUMNS = ['순번', '페이지', '구분', '제목', '지자체명', '서비스세부내용',
           '표_지원주기', '표_신청방법', '표_제공유형',
           '지원대상', '선정기준', '서비스 내용', '신청방법',
           '전화문의', '근거법령', '서식자료', '최종수정일', '상세URL']

# Chrome 설정
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

print(f"\n⚙️ 설정:")
print(f"   디버그 모드: {DEBUG_MODE}")
print(f"   테스트 모드: {TEST_MODE} (1개 항목만)")
print(f"   페이지: {START_PAGE}~{END_PAGE}")
print("="*70)

driver = webdriver.Chrome(options=chrome_options)
collected_data = []

try:
    for page in range(START_PAGE, END_PAGE + 1):
        print(f"\n📄 {page}페이지 처리 중...")

        url = f"{base_url}?page={page}&orderBy=date&tabId={TAB_ID}&period=%EC%B2%AD%EB%85%84"
        driver.get(url)
        time.sleep(5)

        button_xpath = "//a[contains(@aria-label, '자세히 보기')]"

        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.XPATH, button_xpath))
            )
            buttons = driver.find_elements(By.XPATH, button_xpath)
        except:
            print(f"❌ {page}페이지에서 버튼을 찾을 수 없습니다")
            continue

        items_to_process = 1 if TEST_MODE else len(buttons)
        print(f"   총 {len(buttons)}개 중 {items_to_process}개 처리")

        for i in range(items_to_process):
            try:
                current_buttons = driver.find_elements(By.XPATH, button_xpath)
                if i >= len(current_buttons):
                    break

                button = current_buttons[i]
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
                time.sleep(0.5)

                print(f"\n   [{i+1}/{items_to_process}] 클릭...")
                driver.execute_script("arguments[0].click();", button)
                time.sleep(3)

                # 정보 추출
                detail_data = extract_jijache_info(driver, debug=DEBUG_MODE)

                if detail_data and detail_data.get('제목'):
                    detail_data['페이지'] = page
                    detail_data['순번'] = len(collected_data) + 1
                    collected_data.append(detail_data)

                    # 결과 요약
                    filled = sum(1 for v in detail_data.values() if v)
                    print(f"   ✅ {detail_data['제목'][:40]}")
                    print(f"      수집: {filled}/{len(COLUMNS)}개 필드")
                else:
                    print(f"   ❌ 정보 추출 실패")

                # 돌아가기
                driver.back()
                time.sleep(3)

            except Exception as e:
                print(f"   ⚠️ 오류: {type(e).__name__}")
                try:
                    driver.get(url)
                    time.sleep(5)
                except:
                    break

    # 저장
    if collected_data:
        print(f"\n{'='*70}")
        print(f"💾 데이터 저장 중...")
        filename = f'타겟사이트_지자체_{START_PAGE}~{END_PAGE}페이지'
        saved_files = save_data(collected_data, filename, COLUMNS)
        print(f"{'='*70}")
        print(f"🎉 완료! {len(collected_data)}개 항목 저장")
        for f in saved_files:
            print(f"   📁 {f}")
    else:
        print(f"\n⚠️ 수집된 데이터가 없습니다")

except Exception as e:
    print(f"\n❌ 오류 발생: {e}")
    import traceback
    traceback.print_exc()

finally:
    try:
        driver.quit()
    except:
        pass

print(f"\n{'='*70}")
print("프로그램 종료")
print(f"{'='*70}")